## Countries

In [1]:
import pygadm
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pathlib import Path
import pycountry
import os

## Rasters

In [6]:
checkpoint_path = "../results/country_confustion_matrix.parquet"
removed = ["AUS", "BRA", "CAN", "USA", "RUS", "GRL", "MEX", "CHL"]

if os.path.exists(checkpoint_path):
    df = pd.read_parquet(checkpoint_path)
    completed = set(df.gid.to_list())
else:
    df = pd.DataFrame(columns=["country", "gid", "pixel_count", "TN", "FP", "FN", "TP"])
    completed = set()

completed.update(removed)

In [7]:
df

,country,gid,pixel_count,TN,FP,FN,TP
0,Ethiopia,ETH,5347815,5119052,2073,218575,8115
1,Estonia,EST,405438,383109,3532,12895,5902
2,Spain,ESP,3086931,2679750,145495,76600,185086
3,Western Sahara,ESH,1374265,1372862,715,224,464
4,Eritrea,ERI,584115,576126,92,7520,377
...,...,...,...,...,...,...,...
58,Åland Islands,ALA,13830,13470,25,219,116
59,Angola,AGO,5977578,5899914,14557,49634,13473
60,Afghanistan,AFG,3614259,3539717,18928,46755,8859
61,Aruba,ABW,0,0,0,0,0


In [34]:
error_path = "../results/country_errors.parquet"

if os.path.exists(error_path):
    error_df = pd.read_parquet(error_path)
    error = set(error_df.gid.to_list())
    completed.update(error)
else:
    error_df = pd.DataFrame(columns=["country", "gid", "error"])

In [35]:
error_df

,country,gid,error
0,Botswana,BWA,Manifest from NASA DAAC (https://ladsweb.modap...
1,Antarctica,ATA,"Unable to transform edge (296697.215020, 17747..."
2,Anguilla,AIA,y dimension not found. 'rio.set_spatial_dims()...


In [ ]:
import shutil
import numpy as np
from conflict_monitoring_ntl.satellites import BlackMarblePy, GHSLSurface
from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import binarize_xarray, get_combined_mask, get_non_nan_flat_array
from rasterio.enums import Resampling
import datetime
from sklearn.metrics import confusion_matrix

rasters = [GHSLSurface(), BlackMarblePy(frequency="monthly")]
transformations = [{"reproject_match": {"resampling": Resampling.sum}}, {}]
date = datetime.date(2020, 1, 1)

with tqdm(pycountry.countries, desc="Calculating confusion matrix:") as pbar:

    for country_dto in pbar:
        
        country = country_dto.name
        gid = country_dto.alpha_3

        pbar.set_postfix(country=country)

        if gid in completed:
            continue

        try:
            
            gdf = pygadm.Items(admin=gid, content_level=1)
        
            pixels = 0
            conf_mat = np.zeros(4, dtype=np.int64)


            for i in tqdm(range(len(gdf)), desc="Processing Provinces"):

                province_gdf = gpd.GeoDataFrame(gdf.iloc[[i]].geometry).set_crs("EPSG:4326")

                pipeline = RasterPipeline(province_gdf, date, rasters, transformations)
                ds = pipeline.run()

                # make sure we compare non-nan areas
                mask = get_combined_mask(ds)
                ds = ds.where(mask)

                ghsl_pop_binary = binarize_xarray(ds.ghsl_surface, 2500)
                y_true = get_non_nan_flat_array(ghsl_pop_binary)

                bm_binary = binarize_xarray(ds.black_marble_radiance_monthly, 1.0)
                y_pred = get_non_nan_flat_array(bm_binary)

                assert y_pred.shape == y_true.shape

                conf_mat += confusion_matrix(y_true, y_pred).flatten()
                pixels += mask.sum().item()

            data = [country, gid, pixels, *conf_mat.tolist()]
            df = pd.concat([pd.DataFrame([data], columns=df.columns), df], ignore_index=True)
            df.to_parquet(checkpoint_path)

            bm_path = Path(os.path.abspath('')).parent / "data" / "black_marble"
            shutil.rmtree(bm_path)  
            bm_path.mkdir(exist_ok=True)
                
        except Exception as e:

            data = [country, gid, str(e)]
            error_df = pd.concat([pd.DataFrame([data], columns=error_df.columns), error_df], ignore_index=True)
            error_df.to_parquet(error_path)